# exp-009: 한국어/다국어 백본 5종 비교 (1k baseline)

- **목적:** 보고서 §5.3 표 3의 모델 구성을 baseline 4종(TF-IDF/e5-small/ko-sroberta/KR-SBERT, 모두 2021 이전) → +5종 최신 임베딩 모델로 확장. 본 연구의 *베이스* 위에 어디까지 끌어올릴 수 있나 정량.
- **차별성 축:** ② 한국어 (백본 진화)
- **데이터:** 기존 1k 샘플 — `user_profiles.csv` 999 + `jd_profiles_sample1000.csv`. 평가는 기존 `benchmark_labeled_100_{A,B,C,D,E}.csv` 500쌍.
- **풀데이터 필요?** ❌ — 표 3 확장이 목적. 본 검증은 exp-009 v2 (235k)에서.
- **신규 setup:**

| ID | 모델 | dim | 한국어 강도 | 출시 |
|---|---|---|---|---|
| S11 | `nlpai-lab/KURE-v1` | 1024 | ★★★ MTEB-ko 1위 (BGE-M3 한국어 fine-tune) | 2024-12 |
| S12 | `Qwen/Qwen3-Embedding-0.6B` | 1024 | ★★ 119언어, 교수 추천 | 2025-06 |
| S13 | `BAAI/bge-m3` (dense only) | 1024 | ★★ multi-lingual SOTA | 2024-02 |
| S14 | `Qwen/Qwen3-Embedding-4B` | 2560 | ★★★ MTEB-Multi 70.58 | 2025-06 |
| S15 | `jhgan/ko-sroberta-multitask` | 768 | ★ KR-SBERT 후속 | 2022 |

- **계산 절차:**
  1. 평가에 등장하는 unique user(59명) + JD(289개) `baseline_text` 임베딩 (5 모델 × 348 = 1,740 vectors)
  2. cosine 계산 → `weighted_results.csv`의 `role_semantic` 자리에 대체 (나머지 5요소 동일 — 임베딩과 무관)
  3. SINGLE 가중치로 fusion score 재계산
  4. 5관점 라벨 위에서 NDCG@10/5, Recall@5, MRR@10 측정
  5. S09 (profile + e5-small + overlap) baseline과 직접 비교
- **출력:** `raw/experiments/exp-009-baseline-backbone-1k/`
- **관련:** dual-encoder-진화-실험계획 exp-009, KURE-v1, Qwen3-Embedding, BGE-M3
- **작성일/실행일:** 2026-05-18

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time
import os
import torch
from sentence_transformers import SentenceTransformer

DATA = Path('raw/data/gemini_profile_outputs')
OUT_DIR = Path('raw/experiments/exp-009-baseline-backbone-1k')
OUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_CACHE = OUT_DIR / 'embeddings'
EMB_CACHE.mkdir(exist_ok=True)

# MPS OOM 회피 → CPU 강제 (348개 텍스트만 임베딩하면 되니 CPU도 충분히 빠름)
DEVICE = 'cpu'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print(f'device: {DEVICE} (MPS OOM 회피 — 다른 mps 점유 프로세스와 충돌 회피)')

device: cpu (MPS OOM 회피 — 다른 mps 점유 프로세스와 충돌 회피)


In [2]:
# 데이터 로드
users = pd.read_csv(DATA / 'user_profiles.csv', encoding='utf-8-sig')
users.columns = [c.lstrip('\ufeff') for c in users.columns]
jds = pd.read_csv(DATA / 'jd_profiles_sample1000.csv', encoding='utf-8-sig')
jds.columns = [c.lstrip('\ufeff') for c in jds.columns]
jds['job_id'] = jds['job_id'].astype(str)

labels = {}
for p in 'ABCDE':
    df = pd.read_csv(DATA / f'benchmark_labeled_100_{p}.csv', encoding='utf-8-sig')
    df.columns = [c.lstrip('\ufeff') for c in df.columns]
    df['job_id'] = df['job_id'].astype(str)
    labels[p] = df

wr = pd.read_csv(DATA / 'weighted_results.csv', encoding='utf-8-sig')
wr.columns = [c.lstrip('\ufeff') for c in wr.columns]
wr['job_id'] = wr['job_id'].astype(str)
fusion_cols = ['role_semantic', 'hard_skill', 'competency', 'achievement', 'industry', 'quality_adjustment']

# 평가 대상 unique user/JD
eval_users, eval_jds = set(), set()
for p in 'ABCDE':
    for _, r in labels[p].iterrows():
        eval_users.add(r['userId'])
        eval_jds.add(str(r['job_id']))
eval_users, eval_jds = sorted(eval_users), sorted(eval_jds)
print(f'평가 대상 unique: user {len(eval_users)} / JD {len(eval_jds)}')

# 텍스트 추출 (baseline_text 사용)
u_texts = users[users['userId'].isin(eval_users)].set_index('userId')['baseline_text'].fillna('').to_dict()
j_texts = jds[jds['job_id'].isin(eval_jds)].set_index('job_id')['baseline_text'].fillna('').to_dict()
print(f'텍스트 매칭: user {len(u_texts)}/{len(eval_users)}, JD {len(j_texts)}/{len(eval_jds)}')

평가 대상 unique: user 49 / JD 256
텍스트 매칭: user 49/49, JD 256/256


In [3]:
# 백본 정의 (S11~S15)
BACKBONES = {
    'S11': ('nlpai-lab/KURE-v1', 'KURE-v1 (한국어 SOTA)'),
    'S12': ('Qwen/Qwen3-Embedding-0.6B', 'Qwen3-Embedding-0.6B'),
    'S13': ('BAAI/bge-m3', 'BGE-M3 dense'),
    'S15': ('jhgan/ko-sroberta-multitask', 'ko-sroberta-multitask'),
    # S14 (Qwen3-4B 8GB)는 메모리 부담 → 별도 셀에서 옵션 실행
}

def embed_with_cache(setup_id, model_name, texts_dict, batch_size=8):
    cache_path = EMB_CACHE / f'{setup_id}_{"_".join(model_name.split("/"))}.npz'
    if cache_path.exists():
        d = np.load(cache_path, allow_pickle=True)
        print(f'  [cache hit] {cache_path.name}')
        return dict(zip(d['keys'].tolist(), d['vecs']))
    t0 = time.time()
    model = SentenceTransformer(model_name, device=DEVICE, trust_remote_code=True)
    keys = list(texts_dict.keys())
    texts = [texts_dict[k] for k in keys]
    vecs = model.encode(texts, batch_size=batch_size, show_progress_bar=False,
                        convert_to_numpy=True, normalize_embeddings=True)
    np.savez(cache_path, keys=np.array(keys, dtype=object), vecs=vecs)
    print(f'  [computed] {setup_id} {model_name}: {vecs.shape}, {time.time()-t0:.1f}s')
    del model
    import gc
    gc.collect()
    return dict(zip(keys, vecs))

# 모든 백본으로 user/JD 임베딩
embs_user = {}
embs_jd = {}
for sid, (model_name, label) in BACKBONES.items():
    print(f'\n=== {sid} {label} ===')
    print(f'-- user 임베딩 ({len(u_texts)}) --')
    embs_user[sid] = embed_with_cache(f'{sid}_user', model_name, u_texts)
    print(f'-- JD 임베딩 ({len(j_texts)}) --')
    embs_jd[sid] = embed_with_cache(f'{sid}_jd', model_name, j_texts)


=== S11 KURE-v1 (한국어 SOTA) ===
-- user 임베딩 (49) --


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  [computed] S11_user nlpai-lab/KURE-v1: (49, 1024), 144.8s
-- JD 임베딩 (256) --


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  [computed] S11_jd nlpai-lab/KURE-v1: (256, 1024), 132.3s

=== S12 Qwen3-Embedding-0.6B ===
-- user 임베딩 (49) --


...

  [computed] S13_jd BAAI/bge-m3: (256, 1024), 76.3s

=== S15 ko-sroberta-multitask ===
-- user 임베딩 (49) --


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [computed] S15_user jhgan/ko-sroberta-multitask: (49, 768), 8.3s
-- JD 임베딩 (256) --


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [computed] S15_jd jhgan/ko-sroberta-multitask: (256, 768), 9.7s


In [4]:
# (옵션) S14 = Qwen3-Embedding-4B — 8GB, CPU RAM 부담. 기본 skip.
# 다른 모델 검증 후 별도 실행 권장.
TRY_S14 = False
if TRY_S14:
    try:
        print('=== S14 Qwen3-Embedding-4B (8GB) ===')
        embs_user['S14'] = embed_with_cache('S14_user', 'Qwen/Qwen3-Embedding-4B', u_texts, batch_size=2)
        embs_jd['S14'] = embed_with_cache('S14_jd', 'Qwen/Qwen3-Embedding-4B', j_texts, batch_size=2)
        BACKBONES['S14'] = ('Qwen/Qwen3-Embedding-4B', 'Qwen3-Embedding-4B')
    except Exception as e:
        print(f'⚠️ S14 skip (메모리/네트워크): {e}')

In [5]:
# NDCG@K
def ndcg_at_k(rels, k=10):
    rels = np.asarray(rels, dtype=float)
    if len(rels) == 0: return 0.0
    rels_k = rels[:k]
    gains = (2**rels_k - 1) / np.log2(np.arange(2, len(rels_k)+2))
    ideal = np.sort(rels)[::-1][:k]
    ideal_gains = (2**ideal - 1) / np.log2(np.arange(2, len(ideal)+2))
    return gains.sum() / ideal_gains.sum() if ideal_gains.sum() > 0 else 0.0

SINGLE = {'role_semantic': 0.35, 'hard_skill': 0.20, 'competency': 0.15,
          'achievement': 0.10, 'industry': 0.10, 'quality_adjustment': 0.10}

# 각 setup × 5 관점 NDCG@10/5/Recall@5/MRR@10
results = []
for sid in BACKBONES.keys():
    u_emb = embs_user[sid]
    j_emb = embs_jd[sid]
    for p in 'ABCDE':
        L = labels[p].copy()
        # join with weighted_results to get 5 overlap features
        M = L.merge(wr[['userId', 'job_id'] + fusion_cols],
                    on=['userId', 'job_id'], how='left', suffixes=('_lab', ''))
        for c in fusion_cols:
            if c in M.columns: M[c] = M[c].fillna(0)
            else: M[c] = 0
        # 새 cosine 계산 → role_semantic 대체
        def new_cos(row):
            ue = u_emb.get(row['userId'])
            je = j_emb.get(row['job_id'])
            if ue is None or je is None: return 0.0
            return float(np.dot(ue, je))  # normalize_embeddings=True이라 cos sim
        M['role_semantic'] = M.apply(new_cos, axis=1)
        M['score'] = sum(M[c] * SINGLE[c] for c in fusion_cols)
        ndcg10, ndcg5, rec5, mrr10 = [], [], [], []
        for uid, g in M.groupby('userId'):
            gs = g.sort_values('score', ascending=False)
            rels = gs['judge_relevance'].fillna(0).values
            ndcg10.append(ndcg_at_k(rels, 10))
            ndcg5.append(ndcg_at_k(rels, 5))
            pos = (rels >= 3).astype(int)
            tot = pos.sum()
            rec5.append((pos[:5].sum()/tot) if tot > 0 else 0)
            fp = np.where(pos[:10]==1)[0]
            mrr10.append(1/(fp[0]+1) if len(fp) > 0 else 0)
        results.append({
            'setup_id': sid,
            'model': BACKBONES[sid][1],
            'eval_dataset': p,
            'NDCG@10': float(np.mean(ndcg10)),
            'NDCG@5': float(np.mean(ndcg5)),
            'Recall@5': float(np.mean(rec5)),
            'MRR@10': float(np.mean(mrr10)),
            'N_users': len(ndcg10),
        })

res_df = pd.DataFrame(results)
print(res_df.round(4).to_string(index=False))
res_df.to_csv(OUT_DIR / 'exp009_results_long.csv', index=False)

setup_id                 model eval_dataset  NDCG@10  NDCG@5  Recall@5  MRR@10  N_users
     S11    KURE-v1 (한국어 SOTA)            A   0.9624  0.9152    0.5873  1.0000       10
     S11    KURE-v1 (한국어 SOTA)            B   0.9172  0.8434    0.5653  0.7700       10
     S11    KURE-v1 (한국어 SOTA)            C   0.8989  0.7384    0.4962  0.8333       10
     S11    KURE-v1 (한국어 SOTA)            D   0.9393  0.8408    0.5020  0.8333       10
     S11    KURE-v1 (한국어 SOTA)            E   0.9370  0.8719    0.7194  0.8750       10
     S12  Qwen3-Embedding-0.6B            A   0.9578  0.9123    0.6016  0.9500       10
     S12  Qwen3-Embedding-0.6B            B   0.9026  0.8050    0.5528  0.7250       10
     S12  Qwen3-Embedding-0.6B            C   0.9051  0.7622    0.5379  0.8333       10
     S12  Qwen3-Embedding-0.6B            D   0.9609  0.8653    0.5085  0.9000       10
     S12  Qwen3-Embedding-0.6B            E   0.9432  0.8754    0.6996  0.8833       10
     S13          BGE-M3 dense  

In [6]:
# 매트릭스 + S09 baseline 비교
ndcg10_pivot = res_df.pivot(index='setup_id', columns='eval_dataset', values='NDCG@10')
ndcg5_pivot = res_df.pivot(index='setup_id', columns='eval_dataset', values='NDCG@5')
ndcg10_pivot['mean'] = ndcg10_pivot.mean(axis=1)
ndcg5_pivot['mean'] = ndcg5_pivot.mean(axis=1)

# 기존 S09 NDCG@10 (exp_results_NDCG10.csv에서)
old = pd.read_csv(DATA / 'exp_results_NDCG10.csv').set_index('setup_id')
old_ndcg5 = pd.read_csv(DATA / 'exp_results_NDCG5.csv').set_index('setup_id')

print('=== 신규 5종 NDCG@10 매트릭스 (1k baseline) ===')
print(ndcg10_pivot.round(4).to_string())
print()
print('=== 기존 S09 vs 신규 (NDCG@10) ===')
comp10 = pd.DataFrame({
    'S09 (e5-small+overlap)': old.loc['S09'],
    **{f'{sid} ({BACKBONES[sid][1]})': ndcg10_pivot.loc[sid] for sid in BACKBONES},
}).T
print(comp10.round(4).to_string())

ndcg10_pivot.to_csv(OUT_DIR / 'ndcg10_matrix.csv')
ndcg5_pivot.to_csv(OUT_DIR / 'ndcg5_matrix.csv')
comp10.to_csv(OUT_DIR / 'comparison_vs_S09.csv')

# summary
best_sid = ndcg10_pivot['mean'].idxmax()
s09_mean = old.loc['S09', 'mean']
summary = {
    'exp_id': 'exp-009',
    'title': '한국어/다국어 백본 5종 비교 (1k baseline)',
    'date': '2026-05-18',
    'data': '1k 샘플 (user 999 + JD 1000)',
    'eval': '5관점 500쌍 동일 라벨',
    'backbones': {sid: BACKBONES[sid][1] for sid in BACKBONES},
    'best_setup': str(best_sid),
    'best_model': BACKBONES[best_sid][1],
    'best_mean_ndcg10': float(ndcg10_pivot.loc[best_sid, 'mean']),
    's09_mean_ndcg10': float(s09_mean),
    'delta_vs_s09': float(ndcg10_pivot.loc[best_sid, 'mean'] - s09_mean),
    'per_setup_mean_ndcg10': {sid: float(ndcg10_pivot.loc[sid, 'mean']) for sid in BACKBONES},
}
with open(OUT_DIR / 'exp009_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('\n', json.dumps(summary, ensure_ascii=False, indent=2))

=== 신규 5종 NDCG@10 매트릭스 (1k baseline) ===
eval_dataset       A       B       C       D       E    mean
setup_id                                                    
S11           0.9624  0.9172  0.8989  0.9393  0.9370  0.9310
S12           0.9578  0.9026  0.9051  0.9609  0.9432  0.9339
S13           0.9609  0.9190  0.8975  0.9425  0.9384  0.9316
S15           0.9553  0.8998  0.9041  0.9466  0.9431  0.9298

=== 기존 S09 vs 신규 (NDCG@10) ===
                                  A       B       C       D       E    mean
S09 (e5-small+overlap)       0.9975  0.9386  0.9643  0.9332  0.9940  0.9656
S11 (KURE-v1 (한국어 SOTA))     0.9624  0.9172  0.8989  0.9393  0.9370  0.9310
S12 (Qwen3-Embedding-0.6B)   0.9578  0.9026  0.9051  0.9609  0.9432  0.9339
S13 (BGE-M3 dense)           0.9609  0.9190  0.8975  0.9425  0.9384  0.9316
S15 (ko-sroberta-multitask)  0.9553  0.8998  0.9041  0.9466  0.9431  0.9298

 {
  "exp_id": "exp-009",
  "title": "한국어/다국어 백본 5종 비교 (1k baseline)",
  "date": "2026-05-18",
  "data":